### Setup

Imports

In [ ]:
import os
import re
import sys
import importlib
import math
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple, Union
from netCDF4 import Dataset, num2date
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import xarray as xr
from plotly.subplots import make_subplots

Import helper modules from /Tools/helpers

In [ ]:
TOOLS_DIR = Path.cwd().resolve().parent
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

# Read the metadata table
from tools.database_lookup import (
    get_instrument_context,
    resolve_pressure_source_by_id,
)

# Generic helpers
from tools.helpers import clean_label, is_missing, plot_data_by_qc, resolve_good_data_window, safe_tag, save_plotly_figure

# Plotting helpers
from tools.helpers.plot_qa_qc import _to_py_dt

# Pressure sensor comparison helper 
from tools.helpers import plot_pressure_comparison

# Apply Atmospheric Pressure Offset
from tools.helpers import apply_atmospheric_pressure_offset


Definitions

In [ ]:
# Working directory
os.chdir("/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data")

In [ ]:
# Select instrument using ID from satellite_altimetry_moorings_metadata.csv

inst_deploy_id = 270
# 202603:
#           SBE37   SBE26/RBR
#   BASJAS  266     265
#   BASS3A  270     269
#   BASS3B  273     272

database, _row, cfg, metadata = get_instrument_context(
    inst_deploy_id=inst_deploy_id,
    metadata_csv="/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/reference_mooring_proc_info/satellite_altimetry_moorings_metadata.csv",
    print_details=True,
)


Read and create ds, df

In [ ]:
# Read cnv
from tools.parsers.read_sbe37 import read_sbe37_cnv

df = read_sbe37_cnv(cfg["input_file"], verbose=True)

#### Pressure Sensor comparison

In [ ]:
enable_pressure_comparison = True

pressure_inst_deploy_id = 269
# 202603:
#           SBE37   SBE26/RBR
#   BASJAS  266     265
#   BASS3A  270     269
#   BASS3B  273     272

pressure_file = None        # None or path/to/file.ext
comparison_label = None     # Manual override (should be instrument + serial)

# Optional zoom controls
x_start = None
x_end = None

if enable_pressure_comparison:
    fig_press, comp_label_used, comp_file_used = plot_pressure_comparison(
        df=df,
        database=database,
        pressure_inst_deploy_id=pressure_inst_deploy_id,
        pressure_file=pressure_file,
        comparison_label=comparison_label,
        x_start=x_start,
        x_end=x_end,
        y_zoom_to_good=False,
    )
    fig_press.show()

    print(f"Comparison label used: {comp_label_used}")
    print(f"Comparison file used:  {comp_file_used}")
else:
    fig_press = None
    comp_label_used = None
    comp_file_used = pressure_file
    print("Pressure comparison disabled.")

##### Optional: Add/Remove atmospheric pressure to get absolute pressure
Use this when the atmospheric pressure offset has been applied in the instrument setup ONLY.
In this instance (rec_202603 BASJAS SBE37_22467) atmospheric pressure of 10.1325 dbar was applied during sensor's onboard processing, therefore the "PRES" variable is actually relative pressure and not absolute.


In [ ]:
# Sanity Check
print("PRES min/max:", float(np.nanmin(ds["PRES"].values)), float(np.nanmax(ds["PRES"].values)))

In [ ]:
# Atmospheric pressure adjustment (set after inspecting prior plots/QC if needed)
apply_atm_pressure = True      # toggle on/off
atm_offset_dbar = 10.1325      # change this value in-place when needed
atm_mode = "add"               # "add" for relative->absolute in your described case

if apply_atm_pressure:
    ds = apply_atmospheric_pressure_offset(
        ds,
        offset_dbar=atm_offset_dbar,
        pres_var="PRES",
        mode=atm_mode,
        in_place=True,
    )
else:
    print("Atmospheric pressure adjustment skipped.")

In [ ]:
# Sanity Check
print("PRES min/max:", float(np.nanmin(ds["PRES"].values)), float(np.nanmax(ds["PRES"].values)))

In [ ]:
# sync corrected PRES from ds back to df for pressure comparison plotting
df["Pressure [db]"] = ds["PRES"].values


ASHLEY * Check this!!!! ^^^^^

In [ ]:
fig_press, comp_label_used, comp_file_used = plot_pressure_comparison(
        df=df,
        database=database,
        pressure_inst_deploy_id=pressure_inst_deploy_id,
        pressure_file=pressure_file,
        comparison_label=comparison_label,
        x_start=x_start,
        x_end=x_end,
        y_zoom_to_good=False,
    )
fig_press.show()

### QA/QC Flagging

#### Manual QC - set windows and flags

In [ ]:
"""
0: "No_QC_performed", grey-blue
1: "Good_data", blue
2: "Probably_good_data", green
3: "Bad_data_that_are_potentially_correctable", orange
4: "Bad_data", red
5: "Value_changed", purple
6: "Not_used", n/a
7: "Not_used", n/a
8: "Not_used", n/a
9: "Missing_value", white
"""

qc_windows = [
    {"start": "", "end": "2025-02-07 01:55:00", "flag": 4},  # pre-deploy
    {"start": "2026-03-04 00:40:00", "end": "", "flag": 4},  # post-recovery
    # {"start": "2026-03-03 20:50:00", "end": "2026-03-14 01:40:00", "flag": 3},  # following attempted recovery S3B
    # suspected biofouling
    # {"start": "2025-04-07 20:19:00", "end": "2025-04-07 20:22:00", "flag": 4, "qc_vars": ["CNDC_quality_control", "PSAL_quality_control"]},   # S3B
    # {"start": "2026-02-19 06:30:00", "end": "2026-02-19 12:40:00", "flag": 4, "qc_vars": ["CNDC_quality_control", "PSAL_quality_control"]},   # S3B
    # {"start": "2026-02-19 21:10:00", "end": "2026-02-19 23:31:00", "flag": 4, "qc_vars": ["CNDC_quality_control", "PSAL_quality_control"]},   # S3B
    {"start": "2025-08-18 20:39:00", "end": "2025-08-18 20:41:00", "flag": 4, "qc_vars": ["CNDC_quality_control", "PSAL_quality_control"]}, # S3A  
    {"start": "2025-09-17 17:29:00", "end": "2025-09-17 17:31:00", "flag": 4, "qc_vars": ["CNDC_quality_control", "PSAL_quality_control"]},   # S3A
    {"start": "2025-09-18 08:29:00", "end": "2025-09-18 08:31:00", "flag": 4, "qc_vars": ["CNDC_quality_control", "PSAL_quality_control"]},   # S3A 
    {"start": "2025-09-18 09:19:00", "end": "2025-09-18 09:21:00", "flag": 4, "qc_vars": ["CNDC_quality_control", "PSAL_quality_control"]},   # S3A 
]

ds = apply_qc_flag_windows(ds, qc_windows, time_name="TIME")

In [ ]:
plot_data_by_qc(ds)

#### Save Plot

In [ ]:
# Output dir
plot_dir = os.path.abspath(str(cfg["output_dir"]))
os.makedirs(plot_dir, exist_ok=True)

In [ ]:
# Set to None to use database values automatically.
manual_start = "2025-02-07 01:55:00" # None
manual_end = "2026-03-04 00:40:00" # None

zoom_start, zoom_end = resolve_good_data_window(_row, manual_start=manual_start, manual_end=manual_end)
print("Plot window:", zoom_start, "to", zoom_end)

In [ ]:
# 2) Plot only QC=desired
fig = plot_data_by_qc(ds, flags_to_plot=[1], y_zoom_to_good=True, x_start=zoom_start, x_end=zoom_end, legend=None)

# # 3) Per-variable control
# fig = plot_data_by_qc(
#     ds,
#     flags_to_plot={
#         "TEMP": [1,3,4],
#         "CNDC": [1,4],
#         "PSAL": [1,4],
#         "PRES": [1,3],
#     },
#     x_start=zoom_start,
#     x_end=zoom_end,
#     y_zoom_to_good=True,
# )
fig

In [ ]:
plot_dir = os.path.abspath(str(cfg["output_dir"]))
os.makedirs(plot_dir, exist_ok=True)

save_plotly_figure(
    fig,  # <-- use fig (or rename above to fig_qc and use fig_qc here)
    plot_dir,
    f"{cfg['instrument']}_{cfg['serial']}_{cfg['rec_date']}_{cfg['depth']}_post_qc_zoom_to_good.png",
    height=280 * 4
)

In [ ]:
# QA/QC Summary
for qv in [v for v in ds.data_vars if v.endswith("_quality_control")]:
    vals, cnts = np.unique(ds[qv].values, return_counts=True)
    print(qv, dict(zip(vals.tolist(), cnts.tolist())))

* Further modifications to include:
    - Add statistical analysis of the results. i.e. % good, bad, etc

### Save to IMOS-compliant NetCDF


In [ ]:
save_netcdf = True

deliverables_dir = _row.get("imos_deliverables_path")
if isinstance(deliverables_dir, str) and deliverables_dir.lower() == "nan":
    deliverables_dir = None
if not deliverables_dir:
    deliverables_dir = _row.get("imos_path")
if isinstance(deliverables_dir, str) and deliverables_dir.lower() == "nan":
    deliverables_dir = None
if not deliverables_dir:
    deliverables_dir = cfg["output_dir"]

deliverables_dir = os.path.abspath(str(deliverables_dir))
os.makedirs(deliverables_dir, exist_ok=True)

tmp_nc_path = os.path.join(
    deliverables_dir,
    f"{cfg['instrument']}_{cfg['serial']}_{cfg['rec_date']}_{cfg['depth']}_{range_tag}.nc",
)
# Resolve start_of_good_data for filename token
raw_sgd = _row.get("time_coverage_start", None)
if is_missing(raw_sgd):
    sgd = pd.to_datetime(_row.get("deploy_date", None), dayfirst=True, format="mixed", errors="coerce")
    if pd.isna(sgd):
        sgd = pd.to_datetime(time_start)
    time_coverage_start = sgd
else:
    sgd = pd.to_datetime(raw_sgd, dayfirst=True, format="mixed", errors="coerce")
    if pd.isna(sgd):
        sgd = pd.to_datetime(time_start)
    time_coverage_start = sgd

# -----------------------------

# Priority:
#   time_coverage_start -> deploy_date -> dataset start
#   time_coverage_end   -> recovery_date -> dataset end
# -----------------------------
deploy_start = pd.to_datetime(_row.get("time_coverage_start", None), dayfirst=True, format="mixed", errors="coerce")
deploy_end   = pd.to_datetime(_row.get("time_coverage_end", None),   dayfirst=True, format="mixed", errors="coerce")

if pd.isna(deploy_start):
    deploy_start = pd.to_datetime(_row.get("deploy_date", None), dayfirst=True, format="mixed", errors="coerce")
if pd.isna(deploy_end):
    deploy_end = pd.to_datetime(_row.get("recovery_date", None), dayfirst=True, format="mixed", errors="coerce")

if pd.isna(deploy_start):
    deploy_start = pd.to_datetime(time_start)
if pd.isna(deploy_end):
    deploy_end = pd.to_datetime(time_end)

print("time_coverage_start:", time_coverage_start)
print("time_deployment_start:", deploy_start)
print("time_deployment_end:", deploy_end)


In [ ]:
# Trim dataset to deployment window before writing temp nc
# TIME is days since 1950-01-01
t_ds = pd.to_datetime("1950-01-01") + pd.to_timedelta(ds["TIME"].values, unit="D")

mask = (t_ds >= pd.to_datetime(deploy_start)) & (t_ds <= pd.to_datetime(deploy_end))
n_keep = int(mask.sum())

if n_keep == 0:
    raise ValueError(
        f"No samples in deployment window: {deploy_start} to {deploy_end}. "
        f"Dataset span is {t_ds.min()} to {t_ds.max()}."
    )

# keep only deployment period
ds_trimmed = ds.isel(TIME=np.where(mask)[0])

print(f"Trimmed rows: {ds.sizes['TIME']} -> {ds_trimmed.sizes['TIME']}")
print(f"Trimmed range: {t_ds[mask][0]} -> {t_ds[mask][-1]}")

In [ ]:
plot_data_by_qc(ds_trimmed)

In [ ]:

if save_netcdf:
    ds_trimmed.to_netcdf(tmp_nc_path)

    converter = IMOSNetCDFConverter_SBE37(
        input_folder=os.path.dirname(tmp_nc_path),
        input_file=os.path.basename(tmp_nc_path),
        output_dir=deliverables_dir,
    )

    imos_nc_path = converter.process(
        input_nc_path=tmp_nc_path,
        longitude=float(_row["longitude"]),
        latitude=float(_row["latitude"]),
        depth=float(_row["nominal_depth"]),
        inst_channels="CSTZ",
        start_of_good_data=time_coverage_start,
        time_deployment_start=deploy_start,
        time_deployment_end=deploy_end,
        site_code=str(cfg["location"]),
        version=str(_row.get("version", "1")),
        instrument=str(_row["inst_type"]),
        inst_id=str(int(_row["inst_id"])),
        location=cfg["location"],
        metadata_mode="fill_missing",
        nominal_inst_depth=_row.get("nominal_inst_depth", ""),
    )

    print(f"Saved IMOS NetCDF: {imos_nc_path}")

    try:
        os.remove(tmp_nc_path)
    except OSError:
        pass
else:
    print("NetCDF not written. Set save_netcdf = True to save.")